# AsyncIO and aiohttp (with Trading/REST Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to:

- **`asyncio`**: Python’s built-in asynchronous I/O framework
- **`aiohttp`**: an async HTTP client/server library

We focus on **concurrency and parallelizing tasks that take an indeterminate amount
of time**, especially **REST API calls** and combining async I/O with **CPU-bound
work** using executors.

### Contents

- [1. Core Ideas: Event Loop, Tasks, and Awaitables](#1-core-ideas-event-loop-tasks-and-awaitables)
- [2. Basic asyncio Patterns (sleep, gather)](#2-basic-asyncio-patterns-sleep-gather)
- [3. Intro to aiohttp Client](#3-intro-to-aiohttp-client)
- [4. Study Case: Parallel Crypto/Market REST Calls](#4-study-case-parallel-cryptomarket-rest-calls)
- [5. Timeouts, Cancellation, and Error Handling](#5-timeouts-cancellation-and-error-handling)
- [6. Combining AsyncIO with CPU-Bound Work (Executors)](#6-combining-asyncio-with-cpu-bound-work-executors)
- [7. Concurrency Limits and Backpressure](#7-concurrency-limits-and-backpressure)
- [8. Best Practices and Pitfalls](#8-best-practices-and-pitfalls)

## 1. Core Ideas: Event Loop, Tasks, and Awaitables

Key concepts:

- **Event loop**: runs asynchronous tasks and callbacks; there is usually one loop per
  OS thread.
- **Coroutine**: an `async def` function that returns an awaitable when called.
- **Task**: a scheduled coroutine managed by the event loop; created with
  `asyncio.create_task` or `asyncio.gather`.
- **Awaitable**: something you can `await` on (coroutine, `asyncio.Task`, `Future`).

AsyncIO provides **concurrency** by interleaving tasks when they hit an `await` on
I/O or other awaitables. It does *not* run Python bytecode in parallel on multiple
cores by itself; for CPU-bound work you combine it with **executors** (thread or
process pools).

In [12]:
# 1.1 Minimal asyncio example: concurrent sleeps

import asyncio, time
from pandas import Timestamp

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

async def sleepy(name: str, delay: float) -> str:
    t1 = tlog(f"{name} sleeping for {delay}s")
    await asyncio.sleep(delay)
    t2 = tlog(f"{name} woke up")
    return name, (t2 - t1).total_seconds()

async def main_basic():
    t0 = time.perf_counter()
    # Schedule three coroutines concurrently
    results = dict(
        await asyncio.gather(
            sleepy("A", 1.0),
            sleepy("B", 1.5),
            sleepy("C", 0.5),
        )
    )
    elapsed = time.perf_counter() - t0
    print("Results:", results)
    print(f"Total elapsed: {elapsed:.2f}s (max delay ~1.5s)")

# In a notebook, use asyncio.run only once per process; here it's fine for demo
await main_basic()

[18:43:38.676] A sleeping for 1.0s
[18:43:38.676] B sleeping for 1.5s
[18:43:38.676] C sleeping for 0.5s
[18:43:39.179] C woke up
[18:43:39.690] A woke up
[18:43:40.186] B woke up
Results: {'A': 1.013923, 'B': 1.509207, 'C': 0.502978}
Total elapsed: 1.51s (max delay ~1.5s)


## 2. Basic asyncio Patterns (sleep, gather)

Patterns you will use all the time:

- **`asyncio.sleep`**: non-blocking sleep that yields control to the event loop.
- **`asyncio.gather`**: run multiple awaitables concurrently and wait for all.
- **`asyncio.create_task`**: fire-and-forget style scheduling; useful for background
  tasks (e.g. heartbeats, metrics).

We’ll show `gather` for structured concurrency; later sections will use it for
parallel HTTP requests.

In [14]:
# 2.1 Using create_task for background work

import asyncio
from pandas import Timestamp

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

async def heartbeat(interval: float = 1.0):
    while True:
        await asyncio.sleep(interval)
        tlog(f"heartbeat still alive")

async def main_with_heartbeat():
    hb_task = asyncio.create_task(heartbeat(0.7))  # runs until we cancel it
    await asyncio.sleep(2.5)  # simulate other work
    hb_task.cancel()
    try: await hb_task
    except asyncio.CancelledError as EXC:
        tlog(f"Heartbeat cancelled (exc: {EXC!r})")

await main_with_heartbeat()

[18:45:06.737] heartbeat still alive
[18:45:07.452] heartbeat still alive
[18:45:08.167] heartbeat still alive
[18:45:08.526] Heartbeat cancelled (exc: CancelledError())


## 3. Intro to aiohttp Client

`aiohttp` is an async HTTP client/server library built on top of `asyncio`.
We’ll only use the **client** side here.

Core API for GET requests:

```python
import aiohttp

async with aiohttp.ClientSession() as session:
    async with session.get(url, params={...}, timeout=10) as resp:
        data = await resp.json()  # or resp.text(), resp.read()
```

Important points:

- Use a **single `ClientSession`** per process or component and reuse it.
- Use **`async with`** to ensure connections are cleaned up.
- Always **`await`** the response methods (`json()`, `text()`, `read()`).

In [19]:
# 3.1 Simple aiohttp GET to a public crypto endpoint

import asyncio, aiohttp
from pandas import Timestamp

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

# We'll use Binance's public REST API for crypto prices (no auth required).
BINANCE_BASE = "https://api.binance.com"

async def fetch_binance_price(symbol: str, session: aiohttp.ClientSession) -> float | None:
    """Fetch the latest price for a symbol like 'BTCUSDT' from Binance.

    Returns the price as float, or None on error.
    """
    url = f"{BINANCE_BASE}/api/v3/ticker/price"
    params = {"symbol": symbol}
    try:
        async with session.get(url, params=params, timeout=5) as resp:
            resp.raise_for_status()
            data = await resp.json()
            return float(data["price"])
    except Exception as e:
        print(f"Error fetching {symbol}:", e)
        return None

async def main_single_price(symbol: str):
    async with aiohttp.ClientSession() as session:
        return await fetch_binance_price(symbol, session)

symbol = "BTCUSDT"
t1 = tlog(f"Requesting {symbol} price...")
p = await main_single_price(symbol)
t2 = tlog(f"Retrieved {symbol} price: {p:.2f}")
print(f"Delay: {(t2 - t1).total_seconds():.2f}s")

[18:49:29.409] Requesting BTCUSDT price...
[18:49:29.926] Retrieved BTCUSDT price: 65886.43
Delay: 0.52s


## 4. Study Case: Parallel Crypto/Market REST Calls

We now build a small **price fetcher** that:

- Queries multiple symbols in parallel using `aiohttp` + `asyncio.gather`.
- Measures total time vs sequential requests.

We’ll use Binance’s public **`/api/v3/ticker/price`** endpoint which requires **no
authentication** and allows multiple symbols.

In [24]:
# 4.1 Sequential vs concurrent price fetches

import asyncio, aiohttp
from pandas import Timestamp

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT",
          "DOGEUSDT", "XRPUSDT", "ADAUSDT", "LTCUSDT"]

async def fetch_prices_sequential(symbols: list[str]) -> dict[str, float | None]:
    results: dict[str, float | None] = {}
    async with aiohttp.ClientSession() as session:
        for sym in symbols:
            results[sym] = await fetch_binance_price(sym, session)
    return results

async def fetch_prices_concurrent(symbols: list[str]) -> dict[str, float | None]:
    results: dict[str, float | None] = dict()
    tasks: dict[str, asyncio.Task | None] = dict()
    async with aiohttp.ClientSession() as session:
        for symbol in symbols:
            task = asyncio.create_task(
                fetch_binance_price(symbol, session))
            tasks[symbol] = task
        for symbol, task in tasks.items():
            results[symbol] = await task
    return results

async def compare_sequential_vs_concurrent():
    t0 = time.perf_counter()
    seq = await fetch_prices_sequential(SYMBOLS)
    t_seq = time.perf_counter() - t0

    t0 = time.perf_counter()
    conc = await fetch_prices_concurrent(SYMBOLS)
    t_conc = time.perf_counter() - t0

    print("Sequential:")
    print("  ...elapsed =", f"{t_seq:.2f}s")
    print("Concurrent:")
    print("  ...elapsed =", f"{t_conc:.2f}s")
    print("Speedup ~", f"{t_seq/t_conc:.1f}x" if t_conc > 0 else "n/a")
    print("Example prices (concurrent):")
    for symbol in SYMBOLS:
        print(" ", symbol, "->", conc[symbol])

await compare_sequential_vs_concurrent()

Sequential:
  ...elapsed = 3.06s
Concurrent:
  ...elapsed = 0.62s
Speedup ~ 4.9x
Example prices (concurrent):
  BTCUSDT -> 65884.0
  ETHUSDT -> 1985.86
  BNBUSDT -> 609.68
  SOLUSDT -> 82.42
  DOGEUSDT -> 0.09009
  XRPUSDT -> 1.3241
  ADAUSDT -> 0.2465
  LTCUSDT -> 53.79


## 5. Timeouts, Cancellation, and Error Handling

In real trading/market-data code, calls may **hang** or be **slow**. AsyncIO and
`aiohttp` give you tools to:

- Apply **per-request timeouts**.
- Cancel tasks if they exceed a global timeout.
- Handle partial failures while other tasks complete.

We’ll wrap our fetcher with a timeout and show how to use `asyncio.wait_for` and
`asyncio.gather(..., return_exceptions=True)`.

In [30]:
# 5.1 Per-request timeout and handling exceptions in gather

import asyncio
from aiohttp import ClientTimeout, ClientSession
from pandas import Timestamp

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT",
          "DOGEUSDT", "XRPUSDT", "ADAUSDT", "LTCUSDT"]

async def fetch_binance_price(
        symbol: str, session: ClientSession,
        timeout: float = 3) -> float | None:

    timeout = ClientTimeout(total = timeout)
    url = f"{BINANCE_BASE}/api/v3/ticker/price"
    params = {"symbol": symbol}
    args = {"url": url, "params": params, "timeout": timeout}
    try:
        async with session.get(**args) as resp:
            resp.raise_for_status()
            data = await resp.json()
            return float(data["price"])
    except Exception as e:
        print(f"[{symbol}] error:", e)
        return None

async def fetch_all_with_gather(symbols: list[str]) -> dict[str, float | None]:
    async with ClientSession() as session:
        coros = [fetch_binance_price(sym, session) for sym in symbols]
        results_list = await asyncio.gather(*coros, return_exceptions = True)
    out: dict[str, float | None] = {}
    for sym, res in zip(symbols, results_list):
        if isinstance(res, Exception):
            print(f"{sym} failed with exception:", res)
            out[sym] = None
        else:
            out[sym] = res
    return out

t1 = tlog(f"Requesting prices...")
p = await fetch_all_with_gather(SYMBOLS)
t2 = tlog(f"Retrieved prices:")
[print(f" -> {K}: {V}") for K, V in p.items()]
print(f"Delay: {(t2 - t1).total_seconds():.2f}s")

[19:07:59.787] Requesting prices...
[19:08:02.188] Retrieved prices:
 -> BTCUSDT: 65715.9
 -> ETHUSDT: 1980.18
 -> BNBUSDT: 608.59
 -> SOLUSDT: 82.14
 -> DOGEUSDT: 0.08982
 -> XRPUSDT: 1.3208
 -> ADAUSDT: 0.2456
 -> LTCUSDT: 53.57
Delay: 2.40s


In [41]:
# 5.2 Global timeout with asyncio.wait_for

async def fetch_all_with_global_timeout(symbols: list[str], timeout: float = 2.0):
    async with aiohttp.ClientSession() as session:
        coros = [fetch_binance_price(sym, session) for sym in symbols]
        try:
            t0 = time.perf_counter()
            tasks = asyncio.gather(*coros)
            results = await asyncio.wait_for(tasks, timeout = timeout)
            elapsed = time.perf_counter() - t0
            print(f"All done within global timeout: {timeout}s, elapsed: {elapsed:.2f}s")
            return dict(zip(symbols, results))
        except asyncio.TimeoutError:
            elapsed = time.perf_counter() - t0
            print(f"Global timeout {timeout}s hit after {elapsed:.2f}s; tasks cancelled")
            return dict()

await fetch_all_with_global_timeout(SYMBOLS, timeout = 1.0)

All done within global timeout: 1.0s, elapsed: 0.40s


{'BTCUSDT': 65784.85,
 'ETHUSDT': 1982.31,
 'BNBUSDT': 608.99,
 'SOLUSDT': 82.23,
 'DOGEUSDT': 0.08996,
 'XRPUSDT': 1.3224,
 'ADAUSDT': 0.2461,
 'LTCUSDT': 53.69}

## 6. Combining AsyncIO with CPU-Bound Work (Executors)

AsyncIO excels at **I/O-bound** concurrency; for **CPU-bound** code (e.g. a heavy
indicator or risk calculation) you still need multiple threads or processes to
use multiple cores.

The typical pattern:

- Run I/O (HTTP, WebSockets, DB) with `asyncio` / `aiohttp`.
- Offload CPU-bound functions to a **thread pool** or **process pool** using
  `loop.run_in_executor` or `asyncio.to_thread`.

Below we simulate fetching prices asynchronously and then running a heavy CPU
calculation on them in parallel using a `ProcessPoolExecutor`.

In [43]:
# 6.1 CPU-bound function and process pool integration

from concurrent.futures import ProcessPoolExecutor

def cpu_heavy_indicator(prices: list[float]) -> float:
    """Simulate a CPU-heavy indicator calculation (pure Python loop)."""
    total = 0.0
    for p in prices:
        # Some arbitrary math to burn CPU
        total += (p * 0.001) ** 0.5
    return total


async def fetch_prices_for_indicator(symbol: str, session: aiohttp.ClientSession) -> list[float]:
    """Fetch a few recent prices (repeated REST calls) for a symbol."""
    # For demo, fetch the current price N times; in reality you might query a history API
    N = 5
    prices: list[float] = []
    for _ in range(N):
        px = await fetch_binance_price(symbol, session)
        if px is not None:
            prices.append(px)
    return prices


async def run_indicators_with_processpool(symbols: list[str]):
    loop = asyncio.get_running_loop()
    results: dict[str, float] = {}

    async with aiohttp.ClientSession() as session:
        # Step 1: fetch input data concurrently (I/O-bound)
        price_lists = await asyncio.gather(
            *[fetch_prices_for_indicator(sym, session) for sym in symbols]
        )

    # Step 2: run CPU-heavy indicators in a process pool (CPU-bound)
    with ProcessPoolExecutor() as pool:
        tasks = []
        for sym, prices in zip(symbols, price_lists):
            # Offload CPU work to separate process
            tasks.append(loop.run_in_executor(pool, cpu_heavy_indicator, prices))

        indicator_values = await asyncio.gather(*tasks)

    for sym, val in zip(symbols, indicator_values):
        results[sym] = val

    return results


symbols_small = SYMBOLS[:4]
start = time.perf_counter()
vals = await run_indicators_with_processpool(symbols_small)
elapsed = time.perf_counter() - start
print("Indicator values:")
for symbol, v in vals.items():
    print(" ", symbol, "->", v)
print(f"Total elapsed (I/O+CPU with process pool): {elapsed:.2f}s")

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

## 7. Concurrency Limits and Backpressure

When calling public APIs (especially crypto exchanges), you must respect **rate limits**.
AsyncIO makes it very easy to fire hundreds of requests at once — too easy.

Typical patterns to **limit concurrency**:

- Use an **`asyncio.Semaphore`** to cap the number of in-flight requests.
- Schedule tasks in **batches** instead of all at once.

We’ll add a semaphore around our `fetch_binance_price` calls to avoid spamming the API.

In [48]:
# 7.1 Limiting concurrency with a semaphore

from typing import Any

import asyncio, time
from pandas import Timestamp
from asyncio import Semaphore, Task
from aiohttp import ClientSession, ClientTimeout

def tlog(*args):
    t = Timestamp.utcnow()
    ts = t.strftime("%H:%M:%S.%f")
    print(f"[{ts[: -3]}]", *args)
    return t

async def fetch_with_semaphore(symbol: str, session: ClientSession, sem: Semaphore) -> float | None:
    async with sem:  # only N coroutines can enter at once
        return await fetch_binance_price(symbol, session)


async def fetch_all_limited(symbols: list[str], max_concurrent: int = 3) -> dict[str, float | None]:
    sem = Semaphore(max_concurrent)
    tasks = list[Task]()
    async with ClientSession() as session:
        for symbol in symbols:
            task = fetch_with_semaphore(symbol, session, sem)
            tasks.append(asyncio.create_task(task))
        prices = await asyncio.gather(*tasks)
    return dict(zip(symbols, prices))

n = 2
t1 = tlog(f"Getting prices with semaphore (max = {n})")
p = await fetch_all_limited(SYMBOLS, max_concurrent = n)
t2 = tlog("Retrieved prices:")
[print(f" -> {K}: {V}") for K, V in p.items()]
print(f"Delay: {(t2 - t1).total_seconds():.2f}s")

[19:29:53.302] Getting prices with semaphore (max = 2)
[19:29:56.952] Retrieved prices:
 -> BTCUSDT: 65918.87
 -> ETHUSDT: 1990.32
 -> BNBUSDT: 609.9
 -> SOLUSDT: 82.54
 -> DOGEUSDT: 0.09025
 -> XRPUSDT: 1.3274
 -> ADAUSDT: 0.2469
 -> LTCUSDT: 53.85
Delay: 3.65s


## 8. Best Practices and Pitfalls

**Best practices**

- Use **one `ClientSession` per service** and reuse it; creating many sessions is costly.
- Prefer **`asyncio.gather`** for structured concurrency instead of many `create_task`
  without tracking.
- Use **timeouts** (both per-request and global) to avoid hung tasks.
- For CPU-bound work, offload to **executors** (thread/process pools) so the event loop
  stays responsive.
- Implement **concurrency limits** (semaphores) and respect external **rate limits**.

**Pitfalls**

- Mixing **blocking I/O** (e.g. `requests.get`) inside `async def` without offloading:
  this stalls the event loop.
- Calling `asyncio.run` from inside another event loop (e.g. Jupyter); use `await` in
  notebooks or `nest_asyncio` only if you know why.
- Creating too many concurrent tasks at once; use batching or semaphores.
- Forgetting to handle **cancellations** and **exceptions**; `gather(..., return_exceptions=True)`
  can help in fan-out/fan-in patterns.

AsyncIO + aiohttp give you a powerful model for **high-concurrency I/O-bound trading
components** (market data collectors, REST-based execution adapters). Combined with
**executors** for CPU-bound parts, you can keep a single async architecture while
still using multiple cores where it matters.